# Baselines · Fraud random forest and activity gradient boosting
The fraud baseline uses amount and hour in a default random forest. The activity baseline uses standardized sensor readings and the winning system's XGBoost and LightGBM fold configurations. Seed 610 controls the fraud forest and grouped split.

Run top to bottom locally or in Colab. Put `cct_train.zip` and `har_train.csv` beside this notebook, set `DATA_DIR`, or use the project's data layout. Colab prompts for missing files. The final section exports two fitted predictors into `baseline_models`. Use Python 3.12 and the listed packages for the supplied pickles.

In [1]:
# Optional in a fresh kernel: uncomment, run, and restart before continuing.
# %pip install numpy==2.5.5 pandas==3.0.1 scipy==1.18.1 scikit-learn==1.9.1 cloudpickle==3.1.3 xgboost==3.4.1 lightgbm==4.7.0

In [2]:
import os
import sys
import pickle
import zipfile
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

SEED = 610
WORKERS = min(8, os.cpu_count() or 1)
DATA_DIR = Path.cwd()
OUTPUT_DIR = Path.cwd() / "baseline_models"
OUTPUT_DIR.mkdir(exist_ok=True)

def input_path(name):
    for root in (DATA_DIR, DATA_DIR.parent):
        if (root / name).is_file():
            return root / name
        match = next((root / "Docs and Data").rglob(name), None)
        if match:
            return match
    if "google.colab" in sys.modules:
        from google.colab import files
        print("Upload", name)
        files.upload()
        if Path(name).is_file():
            return Path(name)
    raise FileNotFoundError(f"Place {name} in {DATA_DIR} or set DATA_DIR.")

def new_forest():
    return RandomForestClassifier(random_state=SEED, n_jobs=WORKERS)

## 1 · Credit card fraud
Train on transactions before September 1, 2022. Score every later transaction using positive-class F1 and accuracy. Predictors come directly from the payment being scored, so prediction needs no transaction history.

In [3]:
def transaction_features(events):
    return pd.DataFrame({"amount": events["amt"],
                         "hour": events["trans_time"].str.slice(0, 2).astype(int)})

with zipfile.ZipFile(input_path("cct_train.zip")) as archive:
    member, = [name for name in archive.namelist() if Path(name).name == "cct_train.csv"]
    with archive.open(member) as stream:
        fraud = pd.read_csv(stream, usecols=["unix_time", "trans_date", "trans_time", "amt", "is_fraud"])
fraud = fraud.sort_values("unix_time", kind="stable").reset_index(drop=True)
X = transaction_features(fraud)
y = fraud["is_fraud"]
later = pd.to_datetime(fraud.trans_date) >= "2022-09-01"
forest = new_forest().fit(X.loc[~later], y.loc[~later])
pred = forest.predict(X.loc[later])
fraud_result = [accuracy_score(y.loc[later], pred), f1_score(y.loc[later], pred)]

## 2 · Activity recognition with gradient boosting
Identical sensor readings stay in the same fold. Standardization is fitted on each fold's training rows. Three folds use XGBoost and one uses LightGBM. Accuracy and macro F1 pool all held-out predictions.

In [4]:
har = pd.read_csv(input_path("har_train.csv"), header=None)
sensors, activity = har.iloc[:, :-1], har.iloc[:, -1]
groups = pd.util.hash_pandas_object(sensors, index=False).to_numpy()
folds = list(StratifiedGroupKFold(4, shuffle=True, random_state=SEED).split(sensors, activity, groups))

In [5]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def activity_booster(fold):
    """Recreate the previously selected native booster configuration for this fold."""
    if fold == 1:
        return LGBMClassifier(n_estimators=100, num_leaves=4, min_child_samples=16,
                              learning_rate=0.07832503615697724, max_bin=511,
                              colsample_bytree=0.8814460015050225, reg_alpha=0.0009765625,
                              reg_lambda=0.024037654653852953, n_jobs=WORKERS, verbose=-1)
    return XGBClassifier(n_estimators=56, max_leaves=4, max_depth=0, grow_policy="lossguide",
                         tree_method="hist", enable_categorical=True, verbosity=0, random_state=10242048,
                         min_child_weight=2.707170950755067, learning_rate=0.4810290992611119,
                         subsample=0.9659488304727971, colsample_bytree=0.9692046632582698,
                         reg_alpha=0.0009765625, reg_lambda=0.1384884162470523, n_jobs=WORKERS)


# Match the winning booster by fold; numeric label order keeps probabilities aligned.
names, codes = np.unique(activity, return_inverse=True)
booster_oof = np.empty_like(codes)
booster_folds = []
for fold, (train, valid) in enumerate(folds):
    assert not np.intersect1d(groups[train], groups[valid]).size
    model = make_pipeline(StandardScaler(), activity_booster(fold))
    model.fit(sensors.iloc[train].to_numpy(), codes[train])
    booster_oof[valid] = model.predict(sensors.iloc[valid].to_numpy())
    booster_folds.append(model)

booster_result = [accuracy_score(codes, booster_oof), f1_score(codes, booster_oof, average="macro")]

print(pd.DataFrame([["Fraud random forest", *fraud_result, "fraud F1"],
                    ["Activity gradient boosting", *booster_result, "macro F1"]],
                   columns=["Baseline", "Accuracy", "F1", "F1 definition"]).to_string(index=False))

                  Baseline  Accuracy       F1 F1 definition
       Fraud random forest  0.996262 0.470232      fraud F1
Activity gradient boosting  0.787060 0.651842      macro F1


## 3 · Export and predict
Refit the fraud forest on all labeled transactions. For activity prediction, average probabilities from the four trained boosters. Pass fraud transactions with `amt` and `trans_time`, or activity readings with sensor columns in training order.

In [6]:
class BasicFraudPredictor:
    def __init__(self, forest):
        self.forest = forest

    def predict(self, events):
        return self.forest.predict(transaction_features(events))

fraud_model = BasicFraudPredictor(new_forest().fit(X, y))

with (OUTPUT_DIR / "fraud_baseline_random_forest_amount_hour.pkl").open("wb") as stream:
    cloudpickle.dump(fraud_model, stream, protocol=5)
with (OUTPUT_DIR / "fraud_baseline_random_forest_amount_hour.pkl").open("rb") as stream:
    loaded_fraud = pickle.load(stream)
np.testing.assert_array_equal(loaded_fraud.predict(fraud.iloc[:20]), fraud_model.predict(fraud.iloc[:20]))

In [7]:
class BaseActivityBooster:
    def __init__(self, models, labels):
        self.models, self.labels = models, labels

    def predict(self, sensors):
        scores = [model.predict_proba(np.asarray(sensors)) for model in self.models]
        return self.labels[np.mean(scores, axis=0).argmax(axis=1)]

base_activity = BaseActivityBooster(booster_folds, names)
booster_path = OUTPUT_DIR / "activity_baseline_xgboost_lightgbm.pkl"
with booster_path.open("wb") as stream:
    cloudpickle.dump(base_activity, stream, protocol=5)
with booster_path.open("rb") as stream:
    loaded_booster = pickle.load(stream)
np.testing.assert_array_equal(loaded_booster.predict(sensors.iloc[:20]), base_activity.predict(sensors.iloc[:20]))
print("Booster pickle prediction check passed.")

Booster pickle prediction check passed.
